In [1]:
# ============================================================
#  fetch_agmarknet_real.py
#  Fetches REAL commodity market prices from data.gov.in
#  Agmarknet API — All crops, All states, Save to CSV
#
#  STEP 1: Register at https://data.gov.in (free)
#  STEP 2: Go to My Account → API Keys → Copy your key
#  STEP 3: Paste your key in API_KEY below
#  STEP 4: Run → python fetch_agmarknet_real.py
# ============================================================

import requests
import pandas as pd
import time
import os
from datetime import datetime, timedelta

# ============================================================
#  ✅ PASTE YOUR API KEY HERE
#  Get it free from: https://data.gov.in → Register → API Keys
# ============================================================
API_KEY = "579b464db66ec23bdd000001dbf72bc05cae4285765a973a65089d32"

# ============================================================
#  Agmarknet Resource IDs on data.gov.in
#  There are 2 resources — daily prices & variety-wise prices
# ============================================================
RESOURCES = {

    # Variety-wise daily market prices
    "variety_prices": "35985678-0d79-46b4-9ed6-6f13308a1d24",
}

BASE_URL = "https://api.data.gov.in/resource/{resource_id}"

# ── All crops you need ────────────────────────────────────

TARGET_CROPS= ['Rice', 'Maize'  ,'Pegeon Pea(Arhar Fali)',
 'Black Gram Dal(Urd Dal)' ,'Lentil(Masur)(Whole)', 'Pomegranate' ,'Banana' ,'Mango', 'Grapes',
 'Water Melon','Apple', 'Orange', 'Papaya', 'Coconut' ,'Cotton',
 'Jute' ,'Coffee']

'''TARGET_CROPS = [
    "Rice", "Wheat", "Maize", "Cotton", "Soybean",
    "Groundnut", "Sugarcane", "Jowar", "Bajra",
    "Moong", "Urad", "Barley", "Mustard", "Gram",
    "Lentil", "Pea", "Sunflower", "Potato",
    "Watermelon", "Muskmelon", "Cucumber", "Pumpkin"
]'''

# ── All Indian states ─────────────────────────────────────
TARGET_STATES = [
    "Andhra Pradesh", "Arunachal Pradesh",  "Bihar",
    "Chhattisgarh", "Gujarat", "Haryana",
    "Himachal Pradesh","Karnataka", "Kerala",
    "Madhya Pradesh", "Maharashtra", "Odisha", "Punjab", "Rajasthan",
    "Tamil Nadu", "Telangana",
    "Uttar Pradesh", "West Bengal",
    "Delhi"
]

# ============================================================
#  FUNCTION 1: Fetch prices by STATE + COMMODITY
# ============================================================
def fetch_by_state_commodity(state, commodity, limit=500):
    """
    Fetch commodity prices for a specific state and crop.
    Returns list of price records.
    """
    url = BASE_URL.format(resource_id=RESOURCES["variety_prices"])

    params = {
        "api-key":              API_KEY,
        "format":               "json",
        "limit":                limit,
        "filters[State]":       state,
        "filters[Commodity]":   commodity,
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()

        records = data.get("records", [])
        total   = data.get("total", 0)
        print(f"  {state} | {commodity} → {len(records)} records (total: {total})")
        return records

    except requests.exceptions.HTTPError as e:
        print(f"    HTTP Error {state}/{commodity}: {e}")
        return []
    except requests.exceptions.Timeout:
        print(f"    Timeout {state}/{commodity} — skipping")
        return []
    except Exception as e:
        print(f"    Error {state}/{commodity}: {e}")
        return []


# ============================================================
#  FUNCTION 2: Fetch ALL records with PAGINATION
# ============================================================
def fetch_all_pages(state=None, commodity=None,
                    date_from=None, date_to=None,
                    page_size=500):
    """
    Fetch ALL pages of results for given filters.
    Handles pagination automatically.
    """
    url    = BASE_URL.format(resource_id=RESOURCES["variety_prices"])
    offset = 0
    all_records = []

    while True:
        params = {
            "api-key": API_KEY,
            "format":  "json",
            "limit":   page_size,
            "offset":  offset,
        }
        if state:
            params["filters[State]"] = state
        if commodity:
            params["filters[Commodity]"] = commodity
        if date_from:
            params["filters[Arrival_Date][gte]"] = date_from
        if date_to:
            params["filters[Arrival_Date][lte]"] = date_to

        try:
            response = requests.get(url, params=params, timeout=15)
            response.raise_for_status()
            data    = response.json()
            records = data.get("records", [])
            total   = int(data.get("total", 0))

            if not records:
                break

            all_records.extend(records)
            offset += page_size

            print(f"      Page {offset//page_size}: "
                  f"{len(all_records)}/{total} records fetched")

            if offset >= total:
                break

            time.sleep(0.3)  # Respect rate limit

        except Exception as e:
            print(f"  Pagination error at offset {offset}: {e}")
            break

    return all_records


# ============================================================
#  FUNCTION 3: Fetch by DATE RANGE (last N days)
# ============================================================
def fetch_by_date_range(days_back=30, commodity=None, state=None):
    """
    Fetch prices for last N days.
    Great for getting recent market trend data.
    """
    today     = datetime.now()
    date_from = (today - timedelta(days=days_back)).strftime("%d/%m/%Y")
    date_to   = today.strftime("%d/%m/%Y")

    print(f"\n📅 Fetching data from {date_from} to {date_to}")
    if commodity:
        print(f"   Commodity : {commodity}")
    if state:
        print(f"   State     : {state}")

    return fetch_all_pages(
        state=state,
        commodity=commodity,
        date_from=date_from,
        date_to=date_to
    )


# ============================================================
#  FUNCTION 4: Clean & Standardize API Response
# ============================================================
def clean_records(records):
    """
    Clean raw API records into a standard DataFrame.
    """
    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)

    # Rename columns to standard names
    rename_map = {
        "State":            "State",
        "District":         "District",
        "Market":           "Market",
        "Commodity":        "Crop",
        "Variety":          "Variety",
        "Grade":            "Grade",
        "Arrival_Date":     "Date",
        "Min Price":        "Min_Price",
        "Max Price":        "Max_Price",
        "Modal Price":      "Modal_Price",
    }
    df.rename(columns={k: v for k, v in rename_map.items()
                       if k in df.columns}, inplace=True)

    # Convert price columns to numeric
    for col in ["Min_Price", "Max_Price", "Modal_Price"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Parse date
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"],
                                    format="%d/%m/%Y",
                                    errors="coerce")

    # Drop rows with no price
    df.dropna(subset=["Modal_Price"], inplace=True)

    # Remove duplicates
    df.drop_duplicates(inplace=True)

    # Sort by date
    if "Date" in df.columns:
        df.sort_values("Date", ascending=False, inplace=True)

    return df.reset_index(drop=True)


# ============================================================
#  FUNCTION 5: Calculate Price Trend
# ============================================================
def add_price_trend(df):
    """
    Add 30-day price trend and profit metrics to DataFrame.
    """
    import numpy as np

    # Cultivation cost per crop (₹/hectare from CACP)
    CULTIVATION_COST = {
        "Rice": 32000, "Wheat": 28000, "Maize": 22000,
        "Cotton": 45000, "Soybean": 25000, "Groundnut": 35000,
        "Sugarcane": 75000, "Jowar": 18000, "Bajra": 16000,
        "Moong": 20000, "Urad": 20000, "Barley": 20000,
        "Mustard": 24000, "Gram": 22000, "Lentil": 21000,
        "Pea": 28000, "Sunflower": 26000, "Potato": 60000,
        "Watermelon": 40000, "Muskmelon": 38000,
        "Cucumber": 32000, "Pumpkin": 28000,
    }

    if "Crop" not in df.columns or "Modal_Price" not in df.columns:
        return df

    # Group by state+crop, compute trend slope
    trends = []
    for (state, crop), group in df.groupby(["State", "Crop"],
                                            sort=False):
        group = group.sort_values("Date")
        prices = group["Modal_Price"].values

        if len(prices) >= 7:
            x = np.arange(len(prices))
            slope, _ = np.polyfit(x, prices, 1)
            trend_pct = round(slope / prices[0] * 100, 4) \
                        if prices[0] != 0 else 0
        else:
            trend_pct = 0.0

        trend_label = ("Rising"  if trend_pct >  0.05 else
                       "Falling" if trend_pct < -0.05 else
                       "Stable")

        # Profit margin
        modal    = group["Modal_Price"].iloc[-1]
        cost     = CULTIVATION_COST.get(crop, 25000)
        revenue  = modal * 20           # 20 qtl/ha average yield
        profit   = round((revenue - cost) / revenue * 100, 2) \
                   if revenue > 0 else 0

        trends.append({
            "State":           state,
            "Crop":            crop,
            "Price_Trend_30d": trend_pct,
            "Trend_Label":     trend_label,
            "Cultivation_Cost": cost,
            "Profit_Margin":   profit,
        })

    trend_df = pd.DataFrame(trends)
    df = df.merge(trend_df, on=["State", "Crop"], how="left")
    return df


# ============================================================
#  MAIN: Fetch data for all target crops & states
# ============================================================
def fetch_all_market_data(mode="quick"):
    """
    mode = "quick"    → fetch last 30 days for all crops
    mode = "full"     → fetch state x crop combinations
    mode = "single"   → test with one crop+state
    """
    print("\n" + "="*60)
    print("  📈 AGMARKNET DATA FETCHER — data.gov.in")
    print("="*60)

    if API_KEY == "YOUR_API_KEY_HERE":
        print("\n  ❌ ERROR: Please set your API key first!")
        print("  📝 Steps:")
        print("     1. Go to https://data.gov.in")
        print("     2. Register (free) → My Account → API Keys")
        print("     3. Copy key → paste in API_KEY variable above")
        return None

    os.makedirs("raw_data", exist_ok=True)
    all_records = []

    # ── MODE: Quick — fetch last 30 days all crops ─────────
    if mode == "quick":
        print("\n🚀 Mode: QUICK — Last 30 days, all crops")
        for crop in TARGET_CROPS:
            print(f"\n🌾 Fetching: {crop}")
            records = fetch_by_date_range(days_back=30,
                                          commodity=crop)
            all_records.extend(records)
            time.sleep(0.5)

    # ── MODE: Full — state × crop matrix ──────────────────
    elif mode == "full":
        print("\n🚀 Mode: FULL — All states × all crops")
        total = len(TARGET_STATES) * len(TARGET_CROPS)
        count = 0
        for state in TARGET_STATES:
            for crop in TARGET_CROPS:
                count += 1
                print(f"\n[{count}/{total}] {state} | {crop}")
                records = fetch_by_state_commodity(
                    state, crop, limit=200
                )
                all_records.extend(records)
                time.sleep(0.3)

    # ── MODE: Single — test one crop+state ────────────────
    elif mode == "single":
        print("\n🚀 Mode: SINGLE — Test with Rice in Punjab")
        records = fetch_by_state_commodity("Punjab", "Rice",
                                           limit=100)
        all_records.extend(records)

    # ── Clean & save ───────────────────────────────────────
    if not all_records:
        print("\n⚠️  No records fetched. Check API key or filters.")
        return None

    print(f"\n🔄 Cleaning {len(all_records)} raw records...")
    df = clean_records(all_records)

    print("📊 Adding price trends & profit metrics...")
    df = add_price_trend(df)

    output_path = "raw_data/market_prices_real.csv"
    df.to_csv(output_path, index=False)

    print(f"\n✅ Saved → {output_path}")
    print(f"   Records  : {len(df):,}")
    print(f"   Columns  : {list(df.columns)}")
    print(f"   Crops    : {df['Crop'].nunique() if 'Crop' in df.columns else 'N/A'}")
    print(f"   States   : {df['State'].nunique() if 'State' in df.columns else 'N/A'}")

    if "Modal_Price" in df.columns:
        print(f"\n📊 Average Modal Prices (₹/quintal):")
        print(df.groupby("Crop")["Modal_Price"]
              .mean().round(0)
              .sort_values(ascending=False)
              .to_string())

    return df


# ============================================================
#  HOW TO USE — Step by Step Guide
# ============================================================
"""
╔══════════════════════════════════════════════════════════╗
║           HOW TO GET YOUR API KEY (FREE)                 ║
╠══════════════════════════════════════════════════════════╣
║  1. Open browser → https://data.gov.in                   ║
║  2. Click "Sign Up" (top right)                          ║
║  3. Fill name, email, password → Register                ║
║  4. Verify email → Login                                  ║
║  5. Click your name (top right) → "My Account"           ║
║  6. Click "Show API Key" → Copy the key                  ║
║  7. Paste in API_KEY = "your_key_here" above             ║
╚══════════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════════╗
║           HOW TO RUN                                     ║
╠══════════════════════════════════════════════════════════╣
║  # Test with single crop first:                          ║
║  df = fetch_all_market_data(mode="single")               ║
║                                                          ║
║  # Fetch last 30 days for all 22 crops:                  ║
║  df = fetch_all_market_data(mode="quick")                ║
║                                                          ║
║  # Full state × crop matrix (takes ~20 mins):            ║
║  df = fetch_all_market_data(mode="full")                 ║
╚══════════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════════╗
║           OUTPUT CSV COLUMNS                             ║
╠══════════════════════════════════════════════════════════╣
║  State, District, Market, Crop, Variety, Grade,          ║
║  Date, Min_Price, Max_Price, Modal_Price,                ║
║  Price_Trend_30d, Trend_Label,                           ║
║  Cultivation_Cost, Profit_Margin                         ║
╚══════════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════════╗
║           DIRECT API URL (test in browser)               ║
╠══════════════════════════════════════════════════════════╣
║  https://api.data.gov.in/resource/                       ║
║  9ef84268-d588-465a-a308-a864a43d0070                    ║
║  ?api-key=YOUR_KEY                                       ║
║  &format=json                                            ║
║  &limit=10                                               ║
║  &filters[State]=Punjab                                  ║
║  &filters[Commodity]=Rice                                ║
╚══════════════════════════════════════════════════════════╝
"""

if __name__ == "__main__":
    # ── Change mode as needed ─────────────────────────────
    # "single" → test first (1 crop)
    # "quick"  → last 30 days all crops (~5 mins)
    # "full"   → all states × all crops (~20 mins)

    df = fetch_all_market_data(mode="full")

    if df is not None:
        print("\n📋 Sample data:")
        print(df.head(10).to_string(index=False))



  📈 AGMARKNET DATA FETCHER — data.gov.in

🚀 Mode: FULL — All states × all crops

[1/323] Andhra Pradesh | Rice
    Error Andhra Pradesh/Rice: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

[2/323] Andhra Pradesh | Maize
    Error Andhra Pradesh/Maize: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

[3/323] Andhra Pradesh | Pegeon Pea(Arhar Fali)
    Timeout Andhra Pradesh/Pegeon Pea(Arhar Fali) — skipping

[4/323] Andhra Pradesh | Black Gram Dal(Urd Dal)
  Andhra Pradesh | Black Gram Dal(Urd Dal) → 0 records (total: 0)

[5/323] Andhra Pradesh | Lentil(Masur)(Whole)
    Timeout Andhra Pradesh/Lentil(Masur)(Whole) — skipping

[6/323] Andhra Pradesh | Pomegranate
  Andhra Pradesh | Pomegranate → 0 records (total: 0)

[7/323] Andhra Pradesh | Banana
  Andhra Pradesh | Banana → 0 records (total: 0)

[8/323] Andh


  📈 AGMARKNET DATA FETCHER — data.gov.in

🚀 Mode: FULL — All states × all crops

[1/323] Andhra Pradesh | Rice
    Error Andhra Pradesh/Rice: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

[2/323] Andhra Pradesh | Maize
    Error Andhra Pradesh/Maize: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

[3/323] Andhra Pradesh | Pegeon Pea(Arhar Fali)
    Timeout Andhra Pradesh/Pegeon Pea(Arhar Fali) — skipping

[4/323] Andhra Pradesh | Black Gram Dal(Urd Dal)
  Andhra Pradesh | Black Gram Dal(Urd Dal) → 0 records (total: 0)

[5/323] Andhra Pradesh | Lentil(Masur)(Whole)
    Timeout Andhra Pradesh/Lentil(Masur)(Whole) — skipping

[6/323] Andhra Pradesh | Pomegranate
  Andhra Pradesh | Pomegranate → 0 records (total: 0)

[7/323] Andhra Pradesh | Banana
  Andhra Pradesh | Banana → 0 records (total: 0)

[8/323] Andh

KeyboardInterrupt: 